# 3D Attention U-Net pancreas ablation (R0-R4) on Kaggle T4 x2

Inputs: dataset **pancreas-code** (`dist/pancreas-code.zip` from `python scripts/package.py`); optionally a dataset
with preprocessed `cases/*.npz` (output of an earlier run of this notebook) to skip the download.
Settings: Accelerator **GPU T4 x2**, Internet **On**.

Order: **interactive session**: cells 1-5 (setup, data + T1 gate on all 80, benchmark, T6 sanity) -> send the
benchmark JSON and the sanity val-Dice trend to Claude -> then set `RUN_FULL = True` and **Save Version -> Save & Run All
(Commit)** so the full ablation runs unattended (the commit session redoes cells 1-3, ~30 min, unless a prep dataset is attached).

In [ ]:
RUN_SANITY = True     # T6 sanity (interactive). Set False for the commit run.
RUN_FULL   = False    # T7 full ablation R0-R3 (+R4 if WITH_R4). Set True for the commit run.
WITH_R4    = False
WALL_HOURS = 3.0      # wall-clock budget for R0-R3 on 2 GPUs -> benchmark chooses N, K

In [ ]:
# 1. Environment + code + tests
!nvidia-smi
import torch, sys, os, glob, shutil, subprocess, json
print(torch.__version__, 'GPUs:', torch.cuda.device_count(), sys.version)
!pip -q install monai SimpleITK
SRC = glob.glob('/kaggle/input/**/src/train.py', recursive=True)[0].rsplit('/src/', 1)[0]
shutil.copytree(SRC, '/kaggle/working/repo', dirs_exist_ok=True)
%cd /kaggle/working/repo
!python -m src.test_all 2>&1 | tail -22

In [ ]:
# 2. Data: IDC DICOM (public S3) + bundled TCIA labels -> T1 alignment check on ALL 80 (background) + T2 preprocessing
PREP_NPZ = glob.glob('/kaggle/input/**/cases/PANCREAS_0001.npz', recursive=True)
if PREP_NPZ:
    PREP_OUT = PREP_NPZ[0].rsplit('/cases/', 1)[0]
    print('using attached preprocessed data', PREP_OUT)
else:
    PREP_OUT = '/kaggle/working/prep'
    !python tools/fetch_idc.py --meta data/metadata.csv --out /tmp/dicom --workers 32
    t1 = subprocess.Popen('python tools/check_alignment.py --dicom /tmp/dicom --raw_labels data/raw_labels '
                          '--out /kaggle/working/results/alignment_kaggle --overlays 6 > /kaggle/working/t1.log 2>&1', shell=True)
    !python -m src.data.prepare_data --dicom /tmp/dicom --labels data/raw_labels --out {PREP_OUT} --workers 4
    t1.wait()
    shutil.rmtree('/tmp/dicom', ignore_errors=True)
!python tools/unpack_npz.py --src {PREP_OUT} --dst /tmp/prep
!df -h /tmp | tail -1

In [ ]:
# 3. GATES: T1 on all 80 (air ~0, identity rule), T2 QC 80/80, splits identical to the committed file
import pandas as pd
q = json.load(open(f'{PREP_OUT}/qc_summary.json')); print('T2 QC', q)
assert q['n_cases'] == 80 and q['n_pass'] == 80, 'T2 QC failed -> STOP and report'
sp_new, sp_git = json.load(open(f'{PREP_OUT}/patient_splits.json')), json.load(open('/kaggle/working/repo/splits/patient_splits.json'))
assert sp_new == sp_git, 'splits differ from the committed seed-42 split -> STOP'
if not PREP_NPZ:
    s = json.load(open('/kaggle/working/results/alignment_kaggle/alignment_summary.json'))['idc']; print('T1', s)
    r = pd.read_csv('/kaggle/working/results/alignment_kaggle/alignment_report.csv')
    print(r[['before_air_frac', 'before_mean_hu', 'before_in_range_frac', 'before_edge_score']].describe().round(3))
    assert s['rule'] == 'T0_fx0_fy0_fz0' and not s['shape_mismatch'], 'T1 rule differs -> STOP and report'
    assert r.before_air_frac.max() <= 0.01, 'label voxels in air -> STOP and report'
    print('in-range < 0.5 (inspect overlays, expected: low-enhancing pancreas):', s['failing_after'])

In [ ]:
# 4. T5 benchmark -> N, K, 2-GPU schedule (send this JSON to Claude)
!python tools/benchmark_steps.py --data /tmp/prep --wall_hours {WALL_HOURS} --out results/benchmark.json
B = json.load(open('results/benchmark.json')); print(json.dumps(B['recommendation'], indent=1))

In [ ]:
# 5. T6 sanity (~25 min): short R0 on GPU0 and R3 on GPU1, validation only (test set untouched)
if RUN_SANITY:
    s0 = next(r['s_per_step'] for r in B['runs'] if r['run'] == 'R0_baseline')
    S = max(300, int(25 * 60 / s0) // 50 * 50); K = max(50, S // 6 // 50 * 50)
    !python scripts/kaggle_launch.py --data /tmp/prep --sanity_steps {S} --K {K} --out /kaggle/working/sanity --logs /kaggle/working/sanity/logs
    for r in ('R0_baseline', 'R3_final'):
        d = pd.read_csv(f'/kaggle/working/sanity/{r}/train_log.csv').dropna(subset=['val_dice'])
        print(r); print(d[['step', 'train_loss', 'val_dice', 'val_loss', 'lr']].to_string(index=False))

In [ ]:
# 6. T7 full ablation (one process per GPU, resumable: re-running this cell continues from last.pth)
if RUN_FULL:
    !python scripts/kaggle_launch.py --data /tmp/prep --out /kaggle/working/results --logs /kaggle/working/logs {'--with_r4' if WITH_R4 else ''}
    !tail -n 3 /kaggle/working/logs/*.log
    for f in sorted(glob.glob('/kaggle/working/results/R*/summary.json')):
        s = json.load(open(f)); print(s['run'], 'test Dice %.4f +- %.4f' % (s['dice']['mean'], s['dice']['sd']), 'best step', s['best_step'])

In [ ]:
# 7. T8 clinical (C1 volumetry all runs, C2 TTA uncertainty for R3) + T9 figures, from saved outputs
if RUN_FULL:
    !python -m src.clinical --results /kaggle/working/results --data /tmp/prep
    !python tools/make_figures.py --results /kaggle/working/results --data /tmp/prep
    !cp results/benchmark.json /kaggle/working/results/ 2>/dev/null; ls /kaggle/working/results

Monitor while cell 6 runs (in a second notebook tab / after reconnecting): `!tail -n 4 /kaggle/working/logs/*.log` and `!nvidia-smi`.
Download `/kaggle/working/results` (Output tab) as soon as the run ends.